# CommittedAgent vs CommittedAgent Lambda Sweep 0 to 1

This notebook analyzes the current `CommittedAgent` vs `CommittedAgent` 2P3G simulation sweep for `lambda = 0.0 ... 1.0` in steps of `0.1`.

It contains two 4-panel analyses:

- Overall average across all 2P3G distance conditions.
- `equal_to_both` condition only.

The notebook reuses existing simulation JSON files in this directory. It does not rerun simulation.

In [1]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'dataAnalysis').exists()), Path.cwd())
SWEEP_DIR = PROJECT_ROOT / 'dataAnalysis' / 'model_model' / 'committed_agent' / 'outputs' / 'committed_vs_committed_lambda_sweep_0_to_1'
RAW_SWEEP_DIR = PROJECT_ROOT / 'dataAnalysis' / 'raw_data' / 'model_model_simulations' / 'committed_agent' / 'committed_vs_committed_lambda_sweep_0_to_1'
LAMBDAS = [round(i / 10, 1) for i in range(11)]
COND = 'equal_to_both'


def fmt_lambda(lam):
    return str(int(lam)) if float(lam).is_integer() else str(lam).replace('.', 'p')


def mean_ci(values):
    vals = np.asarray([v for v in values if v is not None and not pd.isna(v)], dtype=float)
    if vals.size == 0:
        return np.nan, np.nan, np.nan, 0
    mean = float(np.mean(vals))
    sd = float(np.std(vals, ddof=1)) if vals.size > 1 else 0.0
    se = sd / np.sqrt(vals.size)
    ci = 1.96 * se
    return mean, mean - ci, mean + ci, int(vals.size)


def parse_traj(v):
    return v if isinstance(v, list) else json.loads(v)


def clean_traj(traj):
    out = []
    for p in traj:
        if isinstance(p, list) and len(p) == 2 and (not out or out[-1] != p):
            out.append(p)
    return out


def manhattan(a, b):
    return abs(int(a[0]) - int(b[0])) + abs(int(a[1]) - int(b[1]))

## Overall Average

This section reproduces the 4-panel lambda sweep averaged across all 2P3G distance conditions.

In [2]:
def load_overall_summary():
    rows = []
    for lam in LAMBDAS:
        lam_path = fmt_lambda(lam)
        summary_path = SWEEP_DIR / f'committed_vs_committed_2p3g_summary_lambda_{lam_path}_sessions_0_to_29.json'
        metrics_path = SWEEP_DIR / f'committed_vs_committed_2p3g_notebook_metrics_lambda_{lam_path}_sessions_0_to_29.json'

        summary = json.loads(summary_path.read_text())
        metrics = json.loads(metrics_path.read_text())['summary']

        success_mean, success_low, success_high, success_n = mean_ci(
            [row['successRate'] for row in summary['sessionSummaries']]
        )
        commitment_mean, commitment_low, commitment_high, commitment_n = mean_ci(
            [row['commitmentRate'] for row in summary['sessionSummaries']]
        )
        efficiency = metrics['overall']['efficiency_success_only']
        signaling = metrics['overall']['signaling_all_new_goal']

        rows.append({
            'lambda': lam,
            'total_trials': summary['totalTrials'],
            'success_rate_percent': summary['successRate'] * 100,
            'success_mean_percent': success_mean * 100,
            'success_ci_lower_percent': success_low * 100,
            'success_ci_upper_percent': success_high * 100,
            'success_n_sessions': success_n,
            'coordination_efficiency_percent': efficiency['mean'],
            'coordination_efficiency_ci_lower_percent': efficiency['ci_lower'],
            'coordination_efficiency_ci_upper_percent': efficiency['ci_upper'],
            'coordination_efficiency_n_participants': efficiency['n_participants'],
            'commitment_rate_percent': summary['commitmentRate'] * 100,
            'commitment_mean_percent': commitment_mean * 100,
            'commitment_ci_lower_percent': commitment_low * 100,
            'commitment_ci_upper_percent': commitment_high * 100,
            'commitment_n_sessions': commitment_n,
            'signaling_move_percent': signaling['mean'] * 100,
            'signaling_ci_lower_percent': signaling['ci_lower'] * 100,
            'signaling_ci_upper_percent': signaling['ci_upper'] * 100,
            'signaling_n_participants': signaling['n_participants'],
            'new_goal_trials': summary['newGoalPresentedTrials'],
            'commitment_eligible_trials': summary['commitmentEligibleTrials'],
        })
    return pd.DataFrame(rows)


overall_df = load_overall_summary()
overall_csv = SWEEP_DIR / 'committed_vs_committed_lambda_sweep_0_to_1_summary.csv'
overall_df.to_csv(overall_csv, index=False)
overall_df[['lambda', 'success_rate_percent', 'coordination_efficiency_percent', 'commitment_rate_percent', 'signaling_move_percent']].round(2)

,lambda,success_rate_percent,coordination_efficiency_percent,commitment_rate_percent,signaling_move_percent
0,0.0,93.33,87.93,54.01,31.77
1,0.1,91.39,88.96,69.79,29.10
2,0.2,91.67,89.38,78.84,28.57
3,0.3,90.83,89.21,82.56,28.24
4,0.4,90.83,89.78,85.38,31.40
5,0.5,91.67,89.77,86.50,33.18
6,0.6,91.39,90.03,86.40,33.77
7,0.7,91.39,90.08,86.40,33.60
8,0.8,91.67,90.20,87.34,34.42
9,0.9,91.67,90.18,87.92,35.65


In [3]:
def plot_4panel(df, title, output_path):
    fig, axes = plt.subplots(2, 2, figsize=(13, 10), sharex=True)
    fig.suptitle(title, fontsize=20, fontweight='bold', y=0.98)

    panels = [
        ('Success Rate (%)', 'success_mean_percent', 'success_ci_lower_percent', 'success_ci_upper_percent'),
        ('Coordination Efficiency (%)', 'coordination_efficiency_percent', 'coordination_efficiency_ci_lower_percent', 'coordination_efficiency_ci_upper_percent'),
        ('Commitment (%)', 'commitment_mean_percent', 'commitment_ci_lower_percent', 'commitment_ci_upper_percent'),
        ('Signaling Move (%)', 'signaling_move_percent', 'signaling_ci_lower_percent', 'signaling_ci_upper_percent'),
    ]

    color = '#4f79a8'
    for ax, (panel_title, mean_col, low_col, high_col) in zip(axes.ravel(), panels):
        x = df['lambda'].to_numpy(dtype=float)
        y = df[mean_col].to_numpy(dtype=float)
        low = df[low_col].to_numpy(dtype=float)
        high = df[high_col].to_numpy(dtype=float)
        yerr = np.vstack([y - low, high - y])
        yerr = np.where(np.isfinite(yerr), yerr, 0)

        ax.errorbar(x, y, yerr=yerr, color=color, marker='o', linewidth=2.5, markersize=6, capsize=4)
        ax.set_title(panel_title, fontsize=15, fontweight='bold')
        ax.set_ylim(0, 105)
        ax.set_xlim(-0.03, 1.03)
        ax.set_xticks(LAMBDAS)
        ax.set_xlabel('lambda')
        ax.set_ylabel('(%)')
        ax.grid(axis='y', color='#cfcfcf', linewidth=1.1)
        ax.grid(axis='x', visible=False)
        for spine in ['top', 'right']:
            ax.spines[spine].set_visible(False)

    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(output_path, dpi=200, bbox_inches='tight')
    return fig


overall_png = SWEEP_DIR / 'committed_vs_committed_lambda_sweep_0_to_1_4panel.png'
plot_4panel(overall_df, 'CommittedAgent vs CommittedAgent: Lambda Sweep 0 to 1', overall_png)
plt.show()

/var/folders/wb/_15vsjnj26j9fsl6zcs0cphw0000gn/T/ipykernel_17965/1835195324.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Equal-to-Both Condition Only

This section filters to `distanceCondition == "equal_to_both"` and computes the same four measures.

In [4]:
def compute_efficiency(row, player):
    traj = clean_traj(parse_traj(row[f'player{player}Trajectory']))
    t = row.get('newGoalPresentedTime')
    if t is None or t < 0 or t >= len(traj):
        return None
    start = traj[t]
    actual_steps = (len(traj) - 1) - t
    if actual_steps <= 0:
        return None

    goals = [row.get('target1'), row.get('target2'), row.get('newGoalPosition')]
    goals = [g for g in goals if isinstance(g, list) and len(g) == 2]
    if not goals:
        return None

    optimal_steps = min(manhattan(start, g) for g in goals)
    excess = actual_steps - optimal_steps
    return max(0.0, min(100.0, (1 - excess / actual_steps) * 100))


def compute_signaling(row, player):
    traj = clean_traj(parse_traj(row[f'player{player}Trajectory']))
    t = row.get('newGoalPresentedTime')
    if t is None or t < 0 or t >= len(traj) - 1:
        return None

    before = traj[t]
    after = traj[t + 1]
    shared = row.get('firstDetectedSharedGoal')
    final = row.get(f'player{player}FinalReachedGoal')
    if shared is None or final is None:
        return None

    target1 = row.get('target1')
    target2 = row.get('target2')
    new_goal = row.get('newGoalPosition')
    if shared in (0, 1):
        shared_pos = target1 if shared == 0 else target2
        other_old = target2 if shared == 0 else target1
        new_set = {2}
    elif shared in (1, 2):
        shared_pos = target1 if shared == 1 else target2
        other_old = target2 if shared == 1 else target1
        new_set = {3}
    else:
        return None

    if final in new_set:
        reached_goal_pos = new_goal
        other_goal_pos = shared_pos
    elif final == shared:
        reached_goal_pos = shared_pos
        other_goal_pos = new_goal
    else:
        reached_goal_pos = other_old
        other_goal_pos = new_goal

    moved_closer_to_reached = manhattan(after, reached_goal_pos) < manhattan(before, reached_goal_pos)
    moved_closer_to_other = manhattan(after, other_goal_pos) < manhattan(before, other_goal_pos)
    return 1.0 if moved_closer_to_reached and not moved_closer_to_other else 0.0


def load_equal_to_both_summary():
    rows = []
    for lam in LAMBDAS:
        lam_path = fmt_lambda(lam)
        raw_path = RAW_SWEEP_DIR / f'committed_vs_committed_2p3g_raw_trials_lambda_{lam_path}_sessions_0_to_29.json'
        summary_path = SWEEP_DIR / f'committed_vs_committed_2p3g_summary_lambda_{lam_path}_sessions_0_to_29.json'

        raw = [row for row in json.loads(raw_path.read_text()) if row.get('distanceCondition') == COND]
        summary = json.loads(summary_path.read_text())
        bycond = summary['byCondition'][COND]

        session_trials = {}
        for row in raw:
            session_trials.setdefault(row['sessionIndex'], []).append(row)

        session_success = []
        session_commitment = []
        for session_index in range(30):
            sub = session_trials.get(session_index, [])
            if not sub:
                continue
            session_success.append(sum(1 for row in sub if row.get('collaborationSucceeded')) / len(sub))

            commitments = []
            for row in sub:
                if not row.get('newGoalPresented') or row.get('firstDetectedSharedGoal') is None:
                    continue
                shared = row.get('firstDetectedSharedGoal')
                for player in (1, 2):
                    final = row.get(f'player{player}FinalReachedGoal')
                    if final is not None:
                        commitments.append(1.0 if final == shared else 0.0)
            if commitments:
                session_commitment.append(float(np.mean(commitments)))

        success_mean, success_low, success_high, success_n = mean_ci(session_success)
        commitment_mean, commitment_low, commitment_high, commitment_n = mean_ci(session_commitment)

        participant_rows = {}
        for row in raw:
            if not row.get('newGoalPresented'):
                continue
            for player in (1, 2):
                participant_id = f"session_{row['sessionIndex']}_player{player}"
                participant_rows.setdefault(participant_id, {'efficiency': [], 'signaling': []})

                if row.get('collaborationSucceeded'):
                    efficiency = compute_efficiency(row, player)
                    if efficiency is not None:
                        participant_rows[participant_id]['efficiency'].append(efficiency)

                signaling = compute_signaling(row, player)
                if signaling is not None:
                    participant_rows[participant_id]['signaling'].append(signaling)

        efficiency_values = [np.mean(v['efficiency']) for v in participant_rows.values() if v['efficiency']]
        signaling_values = [np.mean(v['signaling']) for v in participant_rows.values() if v['signaling']]
        efficiency_mean, efficiency_low, efficiency_high, efficiency_n = mean_ci(efficiency_values)
        signaling_mean, signaling_low, signaling_high, signaling_n = mean_ci(signaling_values)

        rows.append({
            'lambda': lam,
            'condition': COND,
            'trials': len(raw),
            'new_goal_trials': sum(1 for row in raw if row.get('newGoalPresented')),
            'success_rate_percent': bycond['successRate'] * 100,
            'success_mean_percent': success_mean * 100,
            'success_ci_lower_percent': success_low * 100,
            'success_ci_upper_percent': success_high * 100,
            'success_n_sessions': success_n,
            'coordination_efficiency_percent': efficiency_mean,
            'coordination_efficiency_ci_lower_percent': efficiency_low,
            'coordination_efficiency_ci_upper_percent': efficiency_high,
            'coordination_efficiency_n_participants': efficiency_n,
            'commitment_rate_percent': bycond['commitmentRate'] * 100,
            'commitment_mean_percent': commitment_mean * 100,
            'commitment_ci_lower_percent': commitment_low * 100,
            'commitment_ci_upper_percent': commitment_high * 100,
            'commitment_n_sessions': commitment_n,
            'signaling_move_percent': signaling_mean * 100,
            'signaling_ci_lower_percent': signaling_low * 100,
            'signaling_ci_upper_percent': signaling_high * 100,
            'signaling_n_participants': signaling_n,
        })
    return pd.DataFrame(rows)


equal_df = load_equal_to_both_summary()
equal_csv = SWEEP_DIR / 'committed_vs_committed_lambda_sweep_0_to_1_equal_to_both_summary.csv'
equal_df.to_csv(equal_csv, index=False)
equal_df[['lambda', 'success_rate_percent', 'coordination_efficiency_percent', 'commitment_rate_percent', 'signaling_move_percent']].round(2)

,lambda,success_rate_percent,coordination_efficiency_percent,commitment_rate_percent,signaling_move_percent
0,0.0,98.89,97.51,48.51,44.17
1,0.1,98.89,97.45,78.17,40.56
2,0.2,97.78,98.34,90.28,44.35
3,0.3,97.78,99.03,94.12,45.00
4,0.4,98.89,99.49,94.85,47.99
5,0.5,98.89,99.49,96.32,48.85
6,0.6,98.89,99.44,96.43,49.43
7,0.7,98.89,99.17,96.38,47.41
8,0.8,98.89,99.17,96.43,48.28
9,0.9,98.89,99.39,96.43,49.14


In [5]:
equal_png = SWEEP_DIR / 'committed_vs_committed_lambda_sweep_0_to_1_equal_to_both_4panel.png'
plot_4panel(equal_df, 'CommittedAgent vs CommittedAgent: Equal-to-Both Only', equal_png)
plt.show()

/var/folders/wb/_15vsjnj26j9fsl6zcs0cphw0000gn/T/ipykernel_17965/909133042.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Output Files

The notebook writes these summary artifacts into the same directory:

- `committed_vs_committed_lambda_sweep_0_to_1_summary.csv`
- `committed_vs_committed_lambda_sweep_0_to_1_4panel.png`
- `committed_vs_committed_lambda_sweep_0_to_1_equal_to_both_summary.csv`
- `committed_vs_committed_lambda_sweep_0_to_1_equal_to_both_4panel.png`